## Write Methods in TimeDB

TimeDB exposes one global write method for multi-series inserts:

| Method | Data shape | Typical source |
|--------|-----------|----------------|
| `td.write()` | Long / tidy — routing info in columns | ETL pipelines, mixed-metric streams |

It:
- Accepts **Pandas or Polars** DataFrames
- Routes every row to its series using a **vectorized Polars join** (no Python loops)
- Converts units via **pint** if needed — per-call (`unit=` kwarg) or per-row (`unit` column)
- Writes all series and all batches in **one transaction**
- Raises `ValueError` *before* any DB write if a series is missing or units are incompatible

The scenario: a 5-turbine wind farm at Gotland, two metrics (`wind_power`, `wind_speed`), 96 × 15-min timestamps.

In [1]:
try:
    import google.colab
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/rebase-energy/timedb/main/examples/colab_setup.py',
        '/tmp/colab_setup.py'
    )
    exec(open('/tmp/colab_setup.py').read())
except ImportError:
    pass  # Not running in Google Colab

In [2]:
import time
import timedb as td
from timedb import TimeSeries
import pandas as pd
import polars as pl
from datetime import datetime, timezone, timedelta

td.delete()
td.create()

Creating database schema...
✓ Schema created successfully


### Setup — create 10 series in one call

`create_series_many()` issues a single round-trip regardless of the number of series.
It is idempotent: re-running returns the existing `series_id` values without error.

In [3]:
TURBINES = [f"T{i:02d}" for i in range(1, 6)]  # T01 … T05
METRICS  = ["wind_power", "wind_speed"]

series_ids = td.create_series_many([
    {"name": metric, "unit": "dimensionless",
     "labels": {"turbine": t, "site": "Gotland"},
     "overlapping": True}
    for t in TURBINES
    for metric in METRICS
])

print(f"Created {len(series_ids)} series — ids: {series_ids}")

Created 10 series — ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


### Build sample data

96 × 15-min steps for one day, 5 turbines × 2 metrics = 960 value pairs.

In [4]:
base_time = datetime(2025, 1, 1, tzinfo=timezone.utc)
steps     = [base_time + timedelta(minutes=15 * i) for i in range(96)]

rows = []
for j, turbine in enumerate(TURBINES):
    for i, t in enumerate(steps):
        rows.append({
            "site":       "Gotland",
            "turbine":    turbine,
            "valid_time": t,
            "wind_power": 50.0 + j * 5 + i * 0.5,
            "wind_speed": 10.0 + j * 0.5 + i * 0.1,
        })

wide_df = pd.DataFrame(rows)

# Unpivot wide → long so each row has a single value
long_df = wide_df.melt(
    id_vars=["site", "turbine", "valid_time"],
    value_vars=["wind_power", "wind_speed"],
    var_name="name",
    value_name="value",
)
print(f"Long-format rows: {len(long_df)}  (2 metrics × 5 turbines × 96 steps)")
print(long_df.head(3).to_string())

Long-format rows: 960  (2 metrics × 5 turbines × 96 steps)
      site turbine                valid_time        name  value
0  Gotland     T01 2025-01-01 00:00:00+00:00  wind_power   50.0
1  Gotland     T01 2025-01-01 00:15:00+00:00  wind_power   50.5
2  Gotland     T01 2025-01-01 00:30:00+00:00  wind_power   51.0


---
## `write()`

The most general shape. Every row encodes its own identity via `name_col` and `label_cols`.

```
name        | site    | turbine | valid_time          | value
------------|---------|---------|---------------------|------
wind_power  | Gotland | T01     | 2025-01-01 00:00    | 50.0
wind_speed  | Gotland | T01     | 2025-01-01 00:00    | 10.0
wind_power  | Gotland | T02     | 2025-01-01 00:00    | 55.0
…
```

`td.write()` resolves all unique `(name, labels)` combinations in **one DB round-trip**,
then writes every partition in **one transaction**.

**Column auto-detection** — you rarely need to spell out `name_col` and `label_cols`:
- `name_col` defaults to `"name"`
- `label_cols` is inferred as every column not in the reserved set
  (`valid_time`, `value`, `knowledge_time`, `change_time`, `valid_time_end`,
  `changed_by`, `annotation`, `series_id`, `batch_id`, `unit`)

Five patterns shown below:

| # | Pattern | How |
|---|---------|-----|
| 1 | Broadcast `knowledge_time` | kwarg |
| 2 | Per-row `knowledge_time` | column |
| 3 | Corrections with `change_time` | two columns |
| 4 | Multiple batches per call | `batch_cols` |
| 5 | Per-row unit conversion | `unit` column |

### Pattern 1 — broadcast `knowledge_time` (kwarg)

All rows share the same `knowledge_time`, passed as a keyword argument.

In [5]:
knowledge_time = datetime(2025, 1, 1, 6, tzinfo=timezone.utc)

t0 = time.perf_counter()
results = td.write(long_df, knowledge_time=knowledge_time)
elapsed = time.perf_counter() - t0

batch_id = results[0].batch_id
print(f"\nInserted {len(results)} series in {elapsed:.3f} s  —  all share batch_id {batch_id}")


Inserted 10 series in 0.071 s  —  all share batch_id 019d1b8f-85e3-7548-8fc1-6c522881df80


### Pattern 2 — per-row `knowledge_time` (column)

Include `knowledge_time` as a column to assign a different value to each row.
The `knowledge_time=` kwarg must be omitted when the column is present.

In [6]:
kt_morning = datetime(2025, 1, 1,  6, tzinfo=timezone.utc)
kt_evening = datetime(2025, 1, 1, 18, tzinfo=timezone.utc)

rows_kt = []
for j, turbine in enumerate(TURBINES):
    for i, t in enumerate(steps):
        rows_kt.append({
            "site":           "Gotland",
            "turbine":        turbine,
            "valid_time":     t,
            "knowledge_time": kt_morning if i < 48 else kt_evening,
            "name":           "wind_power",
            "value":          50.0 + j * 5 + i * 0.5,
        })

long_kt = pd.DataFrame(rows_kt)
print(f"Rows: {len(long_kt)}  —  unique knowledge_times: {long_kt['knowledge_time'].nunique()}")
print(long_kt[["name", "turbine", "valid_time", "knowledge_time", "value"]].head(4).to_string())

results_kt = td.write(long_kt)  # knowledge_time column used per-row; no kwarg
print(f"\nInserted {len(results_kt)} series with per-row knowledge_time")

Rows: 480  —  unique knowledge_times: 2
         name turbine                valid_time            knowledge_time  value
0  wind_power     T01 2025-01-01 00:00:00+00:00 2025-01-01 06:00:00+00:00   50.0
1  wind_power     T01 2025-01-01 00:15:00+00:00 2025-01-01 06:00:00+00:00   50.5
2  wind_power     T01 2025-01-01 00:30:00+00:00 2025-01-01 06:00:00+00:00   51.0
3  wind_power     T01 2025-01-01 00:45:00+00:00 2025-01-01 06:00:00+00:00   51.5

Inserted 5 series with per-row knowledge_time


### Pattern 3 — `knowledge_time` + `change_time` (corrections)

Include both columns to stamp when corrections were applied. On upserts
(`series_id + valid_time` already exists) PostgreSQL's `DEFAULT now()` only fires on
INSERT; TimeDB forces `change_time = now()` in the UPDATE clause unless you supply it.

In [7]:
kt_correction  = datetime(2025, 3, 2, 10, tzinfo=timezone.utc)
correction_time = datetime(2025, 3, 2,  9, tzinfo=timezone.utc)

correction_df = pd.DataFrame([
    {
        "name":           "wind_power",
        "site":           "Gotland",
        "turbine":        "T01",
        "valid_time":     base_time + timedelta(minutes=15 * i),
        "value":          52.0 + i * 0.5,
        "knowledge_time": kt_correction,
        "change_time":    correction_time,
    }
    for i in range(10)
])

print(f"Correcting {len(correction_df)} rows — both knowledge_time and change_time set per-row")
print(correction_df[["valid_time", "value", "knowledge_time", "change_time"]].head(3).to_string())

results_corr = td.write(correction_df)  # both columns passed through per-row; no kwargs needed
print(f"\nUpserted {len(results_corr)} series; change_time={correction_time} stored in DB")

Correcting 10 rows — both knowledge_time and change_time set per-row
                 valid_time  value            knowledge_time               change_time
0 2025-01-01 00:00:00+00:00   52.0 2025-03-02 10:00:00+00:00 2025-03-02 09:00:00+00:00
1 2025-01-01 00:15:00+00:00   52.5 2025-03-02 10:00:00+00:00 2025-03-02 09:00:00+00:00
2 2025-01-01 00:30:00+00:00   53.0 2025-03-02 10:00:00+00:00 2025-03-02 09:00:00+00:00

Upserted 1 series; change_time=2025-03-02 09:00:00+00:00 stored in DB


### Pattern 4 — multiple batches per call (`batch_cols`)

`batch_cols` declares which DataFrame columns define **batch identity** (data provenance).
Each unique combination of values in those columns becomes a distinct batch record.

**With unreserved columns** (e.g. `model`) — values are packed into `batch_params` JSON on
each batch record:

```
name        | site    | turbine | model | valid_time          | value
------------|---------|---------|-------|---------------------|------
wind_power  | Gotland | T01     | ECMWF | 2025-01-01 00:00    | 50.0
wind_power  | Gotland | T01     | GFS   | 2025-01-01 00:00    | 48.5
wind_power  | Gotland | T02     | ECMWF | 2025-01-01 00:00    | 55.0
wind_power  | Gotland | T02     | GFS   | 2025-01-01 00:00    | 53.0
…
```

Two model runs, five turbines, one `write()` call, one transaction — two separate batch
records in `batches_table`, each stamped with `batch_params = {"model": "<name>"}`.

**With reserved columns** (`workflow_id`, `batch_start_time`, `batch_finish_time`) — values
are written **directly to the native `batches_table` fields**, not packed into `batch_params`.
This gives each batch its own queryable provenance identity:

```
name        | site    | turbine | workflow_id       | valid_time          | value
------------|---------|---------|-------------------|---------------------|------
wind_power  | Gotland | T01     | run-2025-01-01-06 | 2025-01-01 00:00    | 50.0
wind_power  | Gotland | T01     | run-2025-01-01-18 | 2025-01-01 12:00    | 51.5
…
```

Two workflow runs written atomically — each row naturally belongs to its run's batch.

In [8]:
MODELS = ["ECMWF", "GFS"]
MODEL_KT = {
    "ECMWF": datetime(2025, 1, 1,  6, tzinfo=timezone.utc),
    "GFS":   datetime(2025, 1, 1,  7, tzinfo=timezone.utc),
}

rows_multi_batch = []
for model in MODELS:
    for j, turbine in enumerate(TURBINES):
        for i, t in enumerate(steps[:24]):  # 24 steps for brevity
            rows_multi_batch.append({
                "name":           "wind_power",
                "site":           "Gotland",
                "turbine":        turbine,
                "model":          model,
                "valid_time":     t,
                "knowledge_time": MODEL_KT[model],
                "value":          50.0 + j * 5 + i * 0.5 + (2.0 if model == "GFS" else 0.0),
            })

multi_batch_df = pd.DataFrame(rows_multi_batch)
print(f"DataFrame: {len(multi_batch_df)} rows — {len(MODELS)} models × {len(TURBINES)} turbines × 24 steps")
print(multi_batch_df[["name", "turbine", "model", "knowledge_time", "valid_time", "value"]].head(4).to_string())

results_mb = td.write(
    multi_batch_df,
    batch_cols=["model"],                          # one batch per model
    workflow_id="nightly-forecast",                # shared across all batches
    # knowledge_time supplied per-row via column — no kwarg needed
)

print(f"\nInserted {len(results_mb)} InsertResult(s)  ({len(MODELS)} batches × {len(TURBINES)} series)")
for r in results_mb:
    print(f"  series_id={r.series_id}  batch_id={r.batch_id}")

DataFrame: 240 rows — 2 models × 5 turbines × 24 steps
         name turbine  model            knowledge_time                valid_time  value
0  wind_power     T01  ECMWF 2025-01-01 06:00:00+00:00 2025-01-01 00:00:00+00:00   50.0
1  wind_power     T01  ECMWF 2025-01-01 06:00:00+00:00 2025-01-01 00:15:00+00:00   50.5
2  wind_power     T01  ECMWF 2025-01-01 06:00:00+00:00 2025-01-01 00:30:00+00:00   51.0
3  wind_power     T01  ECMWF 2025-01-01 06:00:00+00:00 2025-01-01 00:45:00+00:00   51.5



Inserted 10 InsertResult(s)  (2 batches × 5 series)
  series_id=3  batch_id=019d1b8f-8668-7671-8a8b-d0f91f447191
  series_id=9  batch_id=019d1b8f-8668-7671-8a8b-d0f858edab6f
  series_id=5  batch_id=019d1b8f-8668-7671-8a8b-d0f91f447191
  series_id=7  batch_id=019d1b8f-8668-7671-8a8b-d0f858edab6f
  series_id=5  batch_id=019d1b8f-8668-7671-8a8b-d0f858edab6f
  series_id=9  batch_id=019d1b8f-8668-7671-8a8b-d0f91f447191
  series_id=1  batch_id=019d1b8f-8668-7671-8a8b-d0f858edab6f
  series_id=3  batch_id=019d1b8f-8668-7671-8a8b-d0f858edab6f
  series_id=7  batch_id=019d1b8f-8668-7671-8a8b-d0f91f447191
  series_id=1  batch_id=019d1b8f-8668-7671-8a8b-d0f91f447191


In [9]:
rows_two_runs = []
for run_tag, half in [("run-2025-01-01-06", steps[:48]), ("run-2025-01-01-18", steps[48:])]:
    for j, turbine in enumerate(TURBINES[:2]):  # T01, T02 for brevity
        for i, t in enumerate(half[:6]):        # 6 steps per run
            rows_two_runs.append({
                "name":        "wind_power",
                "site":        "Gotland",
                "turbine":     turbine,
                "workflow_id": run_tag,          # reserved — maps to batches_table.workflow_id
                "valid_time":  t,
                "value":       50.0 + j * 5 + i * 0.5,
            })

two_runs_df = pd.DataFrame(rows_two_runs)
print(f"DataFrame: {len(two_runs_df)} rows — 2 workflow runs × 2 turbines × 6 steps")
print(two_runs_df[["turbine", "workflow_id", "valid_time", "value"]].head(4).to_string())

results_runs = td.write(two_runs_df, batch_cols=["workflow_id"])

batch_ids = {r.batch_id for r in results_runs}
print(f"\nInserted {len(results_runs)} InsertResult(s) across {len(batch_ids)} distinct batches")
wf_ids = sorted({r.batch_id for r in results_runs})
print("batch_ids:", wf_ids)

DataFrame: 24 rows — 2 workflow runs × 2 turbines × 6 steps
  turbine        workflow_id                valid_time  value
0     T01  run-2025-01-01-06 2025-01-01 00:00:00+00:00   50.0
1     T01  run-2025-01-01-06 2025-01-01 00:15:00+00:00   50.5
2     T01  run-2025-01-01-06 2025-01-01 00:30:00+00:00   51.0
3     T01  run-2025-01-01-06 2025-01-01 00:45:00+00:00   51.5

Inserted 4 InsertResult(s) across 2 distinct batches
batch_ids: [UUID('019d1b8f-8689-7717-80d0-e88b4b436463'), UUID('019d1b8f-8689-7717-80d0-e88cd468f632')]


### Pattern 5 — per-row unit conversion (`unit` column)

Include a `"unit"` column to apply **different conversion factors to individual rows** in
the same payload. TimeDB resolves the factor for each unique `(series, unit)` combination
via a single vectorized join — pint is invoked only O(unique combos) times.

```
name        | site    | turbine | valid_time          | unit | value
------------|---------|---------|---------------------|------|-------
wind_power  | Gotland | T01     | 2025-01-01 00:00    | MW   |   4.2
wind_power  | Gotland | T01     | 2025-01-01 01:00    | kW   | 4500.0
wind_power  | Gotland | T02     | 2025-01-01 00:00    | MW   |   3.8
…
```

The `unit=` kwarg and the `unit` column are **mutually exclusive** — passing both raises
`ValueError`.

In [10]:
# Create series with a real physical unit for this demo
td.create_series_many([
    {"name": "power_mw", "unit": "MW",
     "labels": {"turbine": t, "site": "Gotland"},
     "overlapping": True, "retention": "medium"}
    for t in TURBINES[:2]  # T01, T02
])

# Build a mixed-unit payload: some rows in MW, some in kW
rows_mixed = []
for j, turbine in enumerate(TURBINES[:2]):
    for i, t in enumerate(steps[:12]):
        incoming_unit = "MW" if i % 2 == 0 else "kW"
        value_mw = 4.0 + j * 0.5 + i * 0.1       # canonical MW value
        value_in = value_mw if incoming_unit == "MW" else value_mw * 1000
        rows_mixed.append({
            "name":       "power_mw",
            "site":       "Gotland",
            "turbine":    turbine,
            "valid_time": t,
            "unit":       incoming_unit,
            "value":      value_in,
        })

mixed_df = pd.DataFrame(rows_mixed)
print(f"Mixed-unit DataFrame: {len(mixed_df)} rows")
print(mixed_df[["turbine", "valid_time", "unit", "value"]].head(6).to_string())

results_unit = td.write(
    mixed_df,
    knowledge_time=datetime(2025, 1, 1, 6, tzinfo=timezone.utc),
    # No unit= kwarg — the column drives per-row conversion
)
print(f"\nInserted {len(results_unit)} series — kW rows converted to MW automatically")

# Verify: read back and confirm stored values are all in MW
ts = td.get_series("power_mw").where(site="Gotland", turbine="T01").read()
stored = ts.to_pandas()
print("\nStored values (should all be in MW, ≈4.0–5.1 range):")
print(stored.head(6).to_string())

Mixed-unit DataFrame: 24 rows
  turbine                valid_time unit   value
0     T01 2025-01-01 00:00:00+00:00   MW     4.0
1     T01 2025-01-01 00:15:00+00:00   kW  4100.0
2     T01 2025-01-01 00:30:00+00:00   MW     4.2
3     T01 2025-01-01 00:45:00+00:00   kW  4300.0
4     T01 2025-01-01 01:00:00+00:00   MW     4.4
5     T01 2025-01-01 01:15:00+00:00   kW  4500.0



Inserted 2 series — kW rows converted to MW automatically



Stored values (should all be in MW, ≈4.0–5.1 range):
                           value
valid_time                      
2025-01-01 00:00:00+00:00    4.0
2025-01-01 00:15:00+00:00    4.1
2025-01-01 00:30:00+00:00    4.2
2025-01-01 00:45:00+00:00    4.3
2025-01-01 01:00:00+00:00    4.4
2025-01-01 01:15:00+00:00    4.5


---
## Read back per-series results

The read API remains single-series. Use `.where()` to filter and read one turbine at a time.

In [11]:
for turbine in TURBINES:
    ts = td.get_series("wind_power").where(site="Gotland", turbine=turbine).read()
    print(f"wind_power / {turbine}: {len(ts)} rows")

wind_power / T01: 96 rows
wind_power / T02: 96 rows
wind_power / T03: 96 rows
wind_power / T04: 96 rows
wind_power / T05: 96 rows


---
## Error handling

`write()` validates series existence, unit compatibility, and column configuration
**before** any DB write. All errors are raised before the transaction opens.

In [12]:
from timedb import IncompatibleUnitError

# ── Missing series ─────────────────────────────────────────────────────────
missing_df = long_df.copy()
missing_df["name"] = missing_df["name"].replace("wind_power", "nonexistent_metric")
try:
    td.write(missing_df, knowledge_time=knowledge_time)
except ValueError as e:
    print(f"Missing series →  ValueError: {e}")

print()

# ── Incompatible unit (kwarg) ──────────────────────────────────────────────
td.create_series("grid_frequency", unit="Hz")
freq_df = pd.DataFrame({
    "metric":     ["grid_frequency"],
    "valid_time": [datetime(2025, 1, 1, tzinfo=timezone.utc)],
    "value":      [50.0],
})
try:
    td.write(freq_df, name_col="metric", knowledge_time=knowledge_time, unit="MW")
except IncompatibleUnitError as e:
    print(f"Incompatible unit kwarg →  IncompatibleUnitError: {e}")

print()

# ── Incompatible unit (column) ─────────────────────────────────────────────
freq_col_df = pd.DataFrame({
    "metric":     ["grid_frequency"],
    "valid_time": [datetime(2025, 1, 1, tzinfo=timezone.utc)],
    "unit":       ["MW"],   # Hz series — MW is incompatible
    "value":      [50.0],
})
try:
    td.write(freq_col_df, name_col="metric", knowledge_time=knowledge_time)
except IncompatibleUnitError as e:
    print(f"Incompatible unit column →  IncompatibleUnitError: {e}")

print()

# ── unit column + unit= kwarg mutual exclusion ─────────────────────────────
try:
    td.write(mixed_df, unit="kW", knowledge_time=knowledge_time)
except ValueError as e:
    print(f"unit col + unit kwarg →  ValueError: {e}")

print()

# ── batch_cols overlapping label_cols ──────────────────────────────────────
try:
    td.write(long_df, label_cols=["site", "turbine"], batch_cols=["turbine"],
             knowledge_time=knowledge_time)
except ValueError as e:
    print(f"batch_cols/label_cols overlap →  ValueError: {e}")

Missing series →  ValueError: No series found for 5 identity combination(s):
  name='nonexistent_metric', labels={'site': 'Gotland', 'turbine': 'T05'}
  name='nonexistent_metric', labels={'site': 'Gotland', 'turbine': 'T03'}
  name='nonexistent_metric', labels={'site': 'Gotland', 'turbine': 'T02'}
  name='nonexistent_metric', labels={'site': 'Gotland', 'turbine': 'T04'}
  name='nonexistent_metric', labels={'site': 'Gotland', 'turbine': 'T01'}

Incompatible unit kwarg →  IncompatibleUnitError: Cannot convert 'MW' to 'Hz'. Units are not dimensionally compatible.

Incompatible unit column →  IncompatibleUnitError: Cannot convert 'MW' to 'Hz'. Units are not dimensionally compatible.

unit col + unit kwarg →  ValueError: Cannot pass both a 'unit' column in the DataFrame and the unit= kwarg. Use the column for per-row units, or the kwarg to broadcast a single unit.

batch_cols/label_cols overlap →  ValueError: batch_cols and label_cols cannot overlap. Column(s) ['turbine'] appear in both. Co